In [1]:
import pulp

def solve_tsp_pulp(distance_matrix):
    num_cities = len(distance_matrix)
    cities = range(num_cities)
    
    # 1. Initialize the Optimization Problem
    prob = pulp.LpProblem("TSP_MTZ", pulp.LpMinimize)
    
    # 2. Define Decision Variables
    # x[i][j] = 1 if the salesman travels directly from city i to city j, else 0
    x = pulp.LpVariable.dicts(
        "x", 
        ((i, j) for i in cities for j in cities), 
        cat=pulp.LpBinary
    )
    
    # u[i] is a continuous dummy variable used to eliminate subtours (MTZ formulation)
    u = pulp.LpVariable.dicts(
        "u", 
        (i for i in cities), 
        lowBound=1, 
        upBound=num_cities, 
        cat=pulp.LpInteger
    )
    
    # 3. Define Objective Function
    # Minimize the total distance traveled
    prob += pulp.lpSum(
        distance_matrix[i][j] * x[i, j] 
        for i in cities for j in cities
    )
    
    # 4. Define Constraints
    # Constraint A: Cannot travel from a city to itself
    for i in cities:
        prob += x[i, i] == 0
        
    # Constraint B: Leave each city exactly once
    for i in cities:
        prob += pulp.lpSum(x[i, j] for j in cities) == 1
        
    # Constraint C: Enter each city exactly once
    for j in cities:
        prob += pulp.lpSum(x[i, j] for i in cities) == 1
        
    # Constraint D: Subtour Elimination (MTZ Formulation)
    # Applies to all pairs except routes involving the starting city (0)
    for i in range(1, num_cities):
        for j in range(1, num_cities):
            if i != j:
                prob += u[i] - u[j] + (num_cities * x[i, j]) <= num_cities - 1

    # 5. Solve the Problem
    # msg=False suppresses the solver output logs
    prob.solve(pulp.PULP_CBC_CMD(msg=False)) 
    
    # 6. Extract the Optimal Tour Order
    tour = [0]
    current_city = 0
    while len(tour) < num_cities:
        for next_city in cities:
            if x[current_city, next_city].varValue and x[current_city, next_city].varValue > 0.9:
                tour.append(next_city)
                current_city = next_city
                break
    tour.append(0) # Return to start
    
    return pulp.value(prob.objective), tour

# --- Example Usage ---
if __name__ == "__main__":
    # Distance matrix between 5 sample cities (0 to 4)
    # 0 represents the starting depot/city
    matrix = [
        [0, 10, 15, 20, 25],
        [10, 0, 35, 25, 18],
        [15, 35, 0, 30, 5],
        [20, 25, 30, 0, 15],
        [25, 18, 5, 15, 0]
    ]
    
    total_dist, optimal_tour = solve_tsp_pulp(matrix)
    
    print(f"Status: Success")
    print(f"Total Distance: {total_dist}")
    print(f"Optimal Tour Path: {' -> '.join(map(str, optimal_tour))}")


Status: Success
Total Distance: 70.0
Optimal Tour Path: 0 -> 2 -> 4 -> 3 -> 1 -> 0
